# 06 Predict

보조 목표는 다음 회차의 후보 보상 상장 여부를 과거 패턴만으로 예측하는 누수 없는 실험 스캐폴드다. maplessunday의 블랙박스 예측과 달리 여기서는 재현 가능한 후보 행렬, 시간순 분할, 베이스라인, 평가 함수만 제공한다. 실제 모델 선택과 학습 코드는 사용자가 직접 채운다.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.predict_dataset import write_prediction_frame

SPLIT_CUTOFF = pd.Timestamp("2025-01-01")

## 데이터 로드

`data/processed/tidy_rewards.csv`와 `data/interim/predict_dataset.csv`만 사용한다. `data/raw`는 읽거나 수정하지 않는다.

후보 상위 N개는 train 구간(`< 2025-01-01`) 빈도로만 선정한다. 테스트 구간 정보가 후보 universe 설계에 들어가지 않도록 같은 컷오프를 후보 선정과 시간순 분할에 함께 사용한다.

In [ ]:
predict_path = ROOT / "data" / "interim" / "predict_dataset.csv"
tidy_path = ROOT / "data" / "processed" / "tidy_rewards.csv"

write_prediction_frame(
    tidy_path=tidy_path,
    output_path=predict_path,
    candidate_cutoff=SPLIT_CUTOFF,
)

df = pd.read_csv(predict_path, parse_dates=["date"])
summary = {
    "행": len(df),
    "회차": df["date"].nunique(),
    "후보": df["reward_norm"].nunique(),
    "양성비율": df["y"].mean(),
    "시작일": df["date"].min().date(),
    "종료일": df["date"].max().date(),
    "후보컷오프": SPLIT_CUTOFF.date(),
}
summary

## 시간순 분할

랜덤 분할은 사용하지 않는다. 기본 컷오프는 `2025-01-01`이며, 학습 구간의 마지막 날짜가 테스트 구간의 첫 날짜보다 앞서는지 확인한다.

In [ ]:
cutoff = SPLIT_CUTOFF
train = df.loc[df["date"] < cutoff].copy()
test = df.loc[df["date"] >= cutoff].copy()

split_summary = {
    "학습행": len(train),
    "테스트행": len(test),
    "학습시작일": train["date"].min().date(),
    "학습종료일": train["date"].max().date(),
    "테스트시작일": test["date"].min().date(),
    "테스트종료일": test["date"].max().date(),
    "학습양성비율": train["y"].mean(),
    "테스트양성비율": test["y"].mean(),
    "분할검증": train["date"].max() < test["date"].min(),
}
assert train["date"].max() < test["date"].min()
split_summary

## 평가 함수

점수는 확률 또는 확률에 준하는 연속값을 받는다. 회차별 적중률은 각 회차의 실제 양성 개수만큼 상위 후보를 골랐을 때 하나 이상 실제 양성을 포함한 회차 비율이다.

점수가 동점이면 `nlargest`가 상위 k개를 고르며, 이 값은 베이스라인 비교를 위한 근사 지표로만 해석한다.

In [ ]:
def evaluate(y_true, y_score, dates):
    y_true = pd.Series(y_true).astype(int).reset_index(drop=True)
    y_score = pd.Series(y_score).astype(float).reset_index(drop=True)
    dates = pd.Series(dates).reset_index(drop=True)
    y_pred = (y_score >= 0.5).astype(int)
    work = pd.DataFrame({"date": dates, "y": y_true, "score": y_score})

    def date_hit(group):
        k = int(group["y"].sum())
        if k < 1:
            return np.nan
        return float(group.nlargest(k, "score")["y"].max())

    auc = roc_auc_score(y_true, y_score) if y_true.nunique() > 1 else np.nan
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": auc,
        "date_hit_rate": work.groupby("date", sort=True).apply(date_hit).mean(),
    }

## 베이스라인

B1은 `days_since_last >= roll_gap_median` 규칙이다. B2는 과거 출현 비율 `past_rate`를 사전확률 점수로 사용한다. 이벤트 지표, 중단 여부, 미래 파생 변수는 사용하지 않는다.

`category`는 보상의 정적 속성으로 보고 전체 이력의 최빈값을 사용하며, 예측 시점에 알려진 값으로 간주한다.

In [ ]:
b1_score = (
    test["roll_gap_median"].notna()
    & (test["days_since_last"] >= test["roll_gap_median"])
).astype(float)
b2_score = test["past_rate"].clip(0, 1)

baseline_results = pd.DataFrame(
    [
        {"기준선": "간격규칙", **evaluate(test["y"], b1_score, test["date"])},
        {"기준선": "사전확률", **evaluate(test["y"], b2_score, test["date"])},
    ]
)
baseline_results

## 모델 입력 준비

`X_train`, `y_train`, `X_test`, `y_test`가 준비되어 있다. 로지스틱 회귀나 트리 계열 모델을 직접 작성하고 `predict_proba` 또는 이에 준하는 점수를 `evaluate()`로 베이스라인과 비교한다.

In [ ]:
feature_columns = [
    "days_since_last",
    "past_count",
    "past_rate",
    "roll_gap_mean",
    "roll_gap_median",
    "lag1_appeared",
    "category",
    "month",
    "quarter",
    "weekofyear",
]

X_all = pd.get_dummies(df[feature_columns], columns=["category"], dummy_na=True)
X_all = X_all.fillna(-1)
X_train = X_all.loc[train.index]
X_test = X_all.loc[test.index]
y_train = train["y"].astype(int)
y_test = test["y"].astype(int)

X_train.shape, X_test.shape, y_train.mean(), y_test.mean()

### 여기에 직접 모델을 작성하세요

## 결론 프레임

이 노트북은 모델 완성이 아니라 누수 없는 비교 환경을 만든다. 직접 작성한 모델이 베이스라인을 넘지 못하면 운영 판단은 과거 패턴 기반 휴리스틱 수준으로 제한해서 해석한다.